# Birdhaus Data Analysis

This notebook provides tools for exploring Birdhaus event, contact, and guest data.

## Setup

First, we load the data from the most recent CSV files.


In [ ]:
import pandas as pd
import glob
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)


In [ ]:
# Data loading utilities

def find_latest_csv(directory: Path, prefix: str) -> Path | None:
    """Find the most recently modified CSV file with the given prefix."""
    pattern = str(directory / f"{prefix}_*.csv")
    files = glob.glob(pattern)
    if not files:
        return None
    return Path(sorted(files, key=lambda x: Path(x).stat().st_mtime, reverse=True)[0])

# Find the project root (handles running from notebooks/ or project root)
notebook_dir = Path('.').resolve()
if notebook_dir.name == 'notebooks':
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
VIEWS_DIR = PROJECT_ROOT / 'data' / 'views'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Views directory: {VIEWS_DIR}")


In [ ]:
# Load raw data
contacts_path = find_latest_csv(DATA_DIR, 'contacts')
events_path = find_latest_csv(DATA_DIR, 'events')
guests_path = find_latest_csv(DATA_DIR, 'guests')

print("Loading raw data:")
print(f"  Contacts: {contacts_path.name if contacts_path else 'NOT FOUND'}")
print(f"  Events: {events_path.name if events_path else 'NOT FOUND'}")
print(f"  Guests: {guests_path.name if guests_path else 'NOT FOUND'}")

contacts = pd.read_csv(contacts_path) if contacts_path else None
events = pd.read_csv(events_path) if events_path else None
guests = pd.read_csv(guests_path) if guests_path else None

print(f"\nLoaded:")
if contacts is not None: print(f"  {len(contacts):,} contacts")
if events is not None: print(f"  {len(events):,} events")
if guests is not None: print(f"  {len(guests):,} guests")


In [ ]:
# Load enriched views (if available)
guests_enriched_path = find_latest_csv(VIEWS_DIR, 'guests_enriched')
contact_history_path = find_latest_csv(VIEWS_DIR, 'contact_event_history')
event_attendance_path = find_latest_csv(VIEWS_DIR, 'event_attendance')

print("Loading enriched views:")
print(f"  Guests Enriched: {guests_enriched_path.name if guests_enriched_path else 'NOT FOUND - run generate_views.py'}")
print(f"  Contact History: {contact_history_path.name if contact_history_path else 'NOT FOUND - run generate_views.py'}")
print(f"  Event Attendance: {event_attendance_path.name if event_attendance_path else 'NOT FOUND - run generate_views.py'}")

guests_enriched = pd.read_csv(guests_enriched_path) if guests_enriched_path else None
contact_history = pd.read_csv(contact_history_path) if contact_history_path else None
event_attendance = pd.read_csv(event_attendance_path) if event_attendance_path else None


---
## Data Overview

Let's explore the structure and content of each dataset.


In [ ]:
# Quick overview of each dataset
print("=" * 60)
print("CONTACTS")
print("=" * 60)
if contacts is not None:
    print(f"Shape: {contacts.shape}")
    print(f"\nColumns: {list(contacts.columns)}")
    display(contacts.head(3))


In [ ]:
print("=" * 60)
print("EVENTS")
print("=" * 60)
if events is not None:
    print(f"Shape: {events.shape}")
    print(f"\nKey columns: event_id, title, status, start_date, primary_category")
    display(events[['event_id', 'title', 'status', 'start_date', 'primary_category']].head(5))


In [ ]:
print("=" * 60)
print("GUESTS")
print("=" * 60)
if guests is not None:
    print(f"Shape: {guests.shape}")
    print(f"\nKey columns: guest_id, event_id, contact_id, guest_type")
    display(guests[['guest_id', 'event_id', 'contact_id', 'guest_type', 'attendance_status']].head(5))


---
## Analysis: Repeat Attendees

Who are the most frequent event attendees?


In [ ]:
# Find repeat attendees using the contact_history view
if contact_history is not None:
    repeat_attendees = contact_history[
        contact_history['unique_events_count'] > 1
    ].sort_values('unique_events_count', ascending=False)
    
    print(f"Repeat attendees (attended more than 1 event): {len(repeat_attendees):,}")
    print(f"\nTop 20 most frequent attendees:")
    display(
        repeat_attendees[[
            'full_name', 'primary_email', 'unique_events_count', 
            'total_attendances', 'is_member'
        ]].head(20)
    )
else:
    print("Contact history view not available. Run generate_views.py first.")


In [ ]:
# Distribution of attendance counts
if contact_history is not None:
    attendance_dist = contact_history['unique_events_count'].value_counts().sort_index()
    print("Attendance distribution (events attended -> number of contacts):")
    print(attendance_dist.head(15))
    
    # Summary stats
    print(f"\nSummary:")
    print(f"  Contacts with 0 events: {(contact_history['unique_events_count'] == 0).sum():,}")
    print(f"  Contacts with 1 event: {(contact_history['unique_events_count'] == 1).sum():,}")
    print(f"  Contacts with 2+ events: {(contact_history['unique_events_count'] >= 2).sum():,}")
    print(f"  Contacts with 5+ events: {(contact_history['unique_events_count'] >= 5).sum():,}")
    print(f"  Contacts with 10+ events: {(contact_history['unique_events_count'] >= 10).sum():,}")


---
## Analysis: Event Popularity

Which events have the most attendees?


In [ ]:
# Most popular events by attendance
if event_attendance is not None:
    popular_events = event_attendance.sort_values('total_guests', ascending=False)
    
    print("Top 20 events by attendance:")
    display(
        popular_events[[
            'title', 'start_date', 'total_guests', 'unique_contacts',
            'buyer_count', 'ticket_holder_count', 'primary_category'
        ]].head(20)
    )
else:
    print("Event attendance view not available. Run generate_views.py first.")


In [ ]:
# Events with no attendees
if event_attendance is not None:
    no_attendees = event_attendance[event_attendance['total_guests'] == 0]
    print(f"Events with no registered guests: {len(no_attendees)}")
    if len(no_attendees) > 0:
        display(no_attendees[['title', 'start_date', 'status', 'primary_category']].head(10))


---
## Analysis: Event Categories

What types of events are most popular?


In [ ]:
# Attendance by event category
if event_attendance is not None:
    category_stats = event_attendance.groupby('primary_category').agg({
        'event_id': 'count',
        'total_guests': 'sum',
        'unique_contacts': 'sum'
    }).rename(columns={
        'event_id': 'event_count',
        'total_guests': 'total_attendances',
        'unique_contacts': 'total_unique_contacts'
    })
    
    category_stats['avg_attendance'] = (
        category_stats['total_attendances'] / category_stats['event_count']
    ).round(1)
    
    category_stats = category_stats.sort_values('total_attendances', ascending=False)
    
    print("Attendance by event category:")
    display(category_stats)


---
## Analysis: Member vs Non-Member Attendance

How do members compare to non-members in event attendance?


In [ ]:
# Member vs non-member comparison
if contact_history is not None:
    member_stats = contact_history.groupby('is_member').agg({
        'contact_id': 'count',
        'unique_events_count': ['sum', 'mean'],
        'total_attendances': ['sum', 'mean']
    }).round(2)
    
    member_stats.columns = [
        'contact_count', 'total_unique_events', 'avg_unique_events',
        'total_attendances', 'avg_attendances'
    ]
    
    print("Member vs Non-Member attendance:")
    display(member_stats)


---
## Analysis: Attendance Over Time

How has attendance changed over time?


In [ ]:
# Attendance trend by month
if event_attendance is not None and 'start_date' in event_attendance.columns:
    ea = event_attendance.copy()
    ea['start_date'] = pd.to_datetime(ea['start_date'])
    ea['month'] = ea['start_date'].dt.to_period('M')
    
    # Only include past events
    past_events = ea[ea['start_date'] < datetime.now()]
    
    monthly_stats = past_events.groupby('month').agg({
        'event_id': 'count',
        'total_guests': 'sum',
        'unique_contacts': 'sum'
    }).rename(columns={
        'event_id': 'events_held',
        'total_guests': 'total_guests',
        'unique_contacts': 'unique_attendees'
    })
    
    print("Monthly attendance (past events only):")
    display(monthly_stats.tail(12))  # Last 12 months


---
## Custom Query Templates

Use these helper functions for your own exploratory queries.


In [ ]:
# Find all events a specific contact attended
def find_contact_events(email_or_name: str):
    """Find all events attended by a contact (search by email or name)."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    # Search by email or name
    mask = (
        guests_enriched['contact_primary_email'].str.contains(email_or_name, case=False, na=False) |
        guests_enriched['contact_full_name'].str.contains(email_or_name, case=False, na=False)
    )
    
    results = guests_enriched[mask][[
        'contact_full_name', 'contact_primary_email',
        'event_title', 'event_start_date', 'event_primary_category',
        'guest_type', 'order_number'
    ]].drop_duplicates()
    
    return results.sort_values('event_start_date', ascending=False)

# Try it:
# find_contact_events("example@email.com")


In [ ]:
# Find all attendees of a specific event
def find_event_attendees(event_title_search: str):
    """Find all attendees of an event (search by title)."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    mask = guests_enriched['event_title'].str.contains(event_title_search, case=False, na=False)
    
    results = guests_enriched[mask][[
        'event_title', 'event_start_date',
        'contact_full_name', 'contact_primary_email',
        'guest_type', 'attendance_status'
    ]].drop_duplicates()
    
    return results.sort_values(['event_title', 'contact_full_name'])

# Try it:
# find_event_attendees("Rope")


In [ ]:
# Find contacts who attended events in a specific category
def find_category_attendees(category: str):
    """Find all unique contacts who attended events in a category."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    # Check which column name exists for category
    cat_col = 'event_category_names' if 'event_category_names' in guests_enriched.columns else 'event_primary_category'
    
    mask = guests_enriched[cat_col].str.contains(category, case=False, na=False)
    
    results = guests_enriched[mask][[
        'contact_full_name', 'contact_primary_email', 'contact_is_member'
    ]].drop_duplicates()
    
    return results.sort_values('contact_full_name')

# Try it:
# find_category_attendees("Education")


---
## Your Custom Queries

Add your own exploration cells below:


In [ ]:
# Your custom queries here:

